# CSE 151B Competition — Optional Notebook Wrapper

**Gradescope verification uses `run_inference()`**, not this notebook:

```bash
python run_inference.py
```

This notebook is optional convenience for smoke tests, validation sweeps, and
recovery experiments. Defaults match [`src/qwen3_comp/inference_config.py`](src/qwen3_comp/inference_config.py).

Run inside a **DSMLP GPU pod** (`bash dsmlp/launch_gpu.sh`).

## 1. Configuration

Edit the flags below, then run this cell once per session.

In [ ]:
from qwen3_comp.inference_config import FINAL_INFERENCE_DEFAULTS
from qwen3_comp.pipeline import PipelineConfig

# Defaults match inference_config.py (same as `python run_inference.py`).
# For Gradescope reproduction, prefer:  python run_inference.py

CFG = PipelineConfig(
    # --- reproduce submission (same as run_inference.py) ---
    run_full_submission=True,        # set True to mirror Gradescope verification
    run_smoke_test=False,
    run_build_validation_split=False,
    run_prepare_recovery=False,
    run_targeted_recovery=False,
    rebuild_csv_only=False,

    # paths / hyperparams inherit FINAL_INFERENCE_DEFAULTS via PipelineConfig
)
print("Config loaded. Final defaults:", FINAL_INFERENCE_DEFAULTS)

## 2. Setup and environment check

In [ ]:
import json
from qwen3_comp.pipeline import setup_notebook, check_environment

REPO_ROOT = setup_notebook()
env = check_environment()
print(json.dumps(env, indent=2))

if not env["cuda_available"]:
    raise RuntimeError("CUDA not available — launch a DSMLP GPU pod first.")
if not env["in_project_venv"]:
    print("WARNING: activate the project venv: source .venv/bin/activate")

## 3. Install environment (first time only)

Run once on a fresh DSMLP pod, then set `CFG.run_install = False`.

In [ ]:
if CFG.run_install:
    from qwen3_comp.pipeline import install_environment
    install_environment()
    print("Install complete. Restart the kernel, re-run cells from Configuration onward.")
else:
    print("Skipped install (CFG.run_install=False).")

## 4. Smoke test (3 public questions)

In [ ]:
if CFG.run_smoke_test:
    from qwen3_comp.pipeline import run_smoke_test
    code = run_smoke_test(n=3)
    if code != 0:
        raise RuntimeError("Smoke test failed — fix environment before full runs.")
else:
    print("Skipped smoke test.")

## 5. Build validation split + run validation (optional)

In [ ]:
if CFG.run_build_validation_split:
    from qwen3_comp.pipeline import build_validation_split
    print(json.dumps(build_validation_split(CFG), indent=2))
else:
    print("Skipped validation split build.")

if CFG.run_validation:
    from qwen3_comp.pipeline import run_validation
    summary = run_validation(CFG)
    print("Validation accuracy:", summary["accuracy"].get("all"))
else:
    print("Skipped validation run.")

## 6. Freeze best submission (do once after your best Kaggle score)

In [ ]:
from qwen3_comp.pipeline import freeze_best_submission

manifest = freeze_best_submission(
    source_csv=CFG.out_csv,
    source_checkpoint="responses.jsonl.backup",
    dest_dir=CFG.best_dir,
    kaggle_score=0.636,
)
print(json.dumps(manifest, indent=2))

## 7. Private submission runs

Choose **one** path:
- `run_full_submission=True` — all 943 rows from scratch
- `run_targeted_recovery=True` — rerun only insane/unboxed rows (recommended after 0.636)
- `rebuild_csv_only=True` — no GPU; rebuild CSV from checkpoint

In [ ]:
from qwen3_comp.pipeline import (
    prepare_rerun_checkpoint,
    run_submission,
    run_targeted_recovery,
)

if CFG.run_prepare_recovery:
    prep = prepare_rerun_checkpoint(CFG)
    print("Rerun checkpoint:", json.dumps(prep, indent=2))

if CFG.run_full_submission:
    report = run_submission(CFG)
    print(json.dumps(report, indent=2))
elif CFG.run_targeted_recovery:
    report = run_targeted_recovery(CFG)
    print(json.dumps(report, indent=2))
elif CFG.rebuild_csv_only:
    CFG.skip_inference = True
    report = run_submission(CFG)
    print(json.dumps(report, indent=2))
else:
    print("No submission step selected.")

## 8. Audit checkpoint + QLoRA gate

In [ ]:
from qwen3_comp.pipeline import audit_checkpoint, evaluate_qlora_gate

audit_path = CFG.rerun_checkpoint if CFG.run_targeted_recovery else CFG.responses_jsonl
audit = audit_checkpoint(audit_path)
print("Checkpoint audit:")
print(json.dumps(audit, indent=2))

gate = evaluate_qlora_gate(audit_path)
print("\nQLoRA gate:")
print(json.dumps(gate, indent=2))

if audit.get("boxed_rate", 0) >= 0.90 and audit.get("sane_rate", 0) >= 0.88:
    print("\nReady for Kaggle upload if validation looks good.")
else:
    print("\nConsider another targeted recovery pass before uploading.")

## 9. One-click: run everything enabled in CFG

Optional convenience cell — runs all steps whose flags are `True` in Cell 2.

In [ ]:
from qwen3_comp.pipeline import run_notebook_pipeline

results = run_notebook_pipeline(CFG)
print(json.dumps({k: v for k, v in results.items() if k != "environment"}, indent=2, default=str))